# KdV — Prediction Error vs Lead Time

Compares **RMSE** and **relative RMSE** at each autoregressive prediction step for all models in `logs/official/kdv_experiments.json`.

Each model is rolled out from `t=0` for up to `T-1` steps on the **test set**. The per-step RMSE is plotted to show how quickly each model's error grows with prediction horizon.

In [ ]:
from pathlib import Path

import numpy as np
import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.plot_config import MODEL_COLORS  # noqa: E402
from notebooks.prediction.helpers import (  # noqa: E402
    discover_models,
    load_test_data,
    plot_rel_rmse,
    print_summary_table,
    run_all_rollouts,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
DATASET_KEY = "kdv"
DATA_DIR = "kdv"
RUNS_BASE = ROOT / "logs/official/runs"
N_EVAL = 10  # number of test trajectories to evaluate

# Models to include in the plot (order is preserved)
PLOT_MODELS = [
    "fno",
    "kdv",
    # "kdv_realV",
    # "res_onsagernet",
    "onsagernet1d_lowrank",
    "s_onsagernet",
]

# Display names for each model
DISPLAY_NAMES = {
    "fno": "FNO",
    "kdv": "KdV (classical solver)",
    "kdv_realV": "SpecOnsNet w/ real $V$",
    "res_onsagernet": "Res-OnsagerNet",
    "onsagernet1d_lowrank": "OnsagerNet",
    "s_onsagernet": "SpecOnsNet (ours)",
}

models = discover_models(RUNS_BASE / DATASET_KEY, root=ROOT, device=device)

test_data, t_coord, x_coord = load_test_data(str(ROOT / f"data/{DATA_DIR}/*.hdf5"))
N_test, T, n_vars, Nx = test_data.shape

sample_idxs = np.linspace(0, N_test - 1, N_EVAL, dtype=int)
eval_data = test_data[sample_idxs]  # (N_EVAL, T, n_vars, Nx)

print(f"Test set : {N_test} trajectories, using {N_EVAL} samples at indices {sample_idxs.tolist()}")
print(f"Shape    : T={T}, n_vars={n_vars}, Nx={Nx}")
print(f"t range  : {t_coord[0]:.4f} -> {t_coord[-1]:.4f}  (dt={t_coord[1] - t_coord[0]:.4f})")

In [ ]:
MAX_STEPS = 100  # number of prediction steps to evaluate

results = run_all_rollouts(models, eval_data, max_steps=MAX_STEPS, device=device)

In [ ]:
REPORT_STEPS = [1, 2, 5]  # prediction steps to report (must be <= MAX_STEPS)
print_summary_table(results, DISPLAY_NAMES, REPORT_STEPS, MAX_STEPS)

In [ ]:
plot_rel_rmse(
    results,
    plot_models=PLOT_MODELS,
    display_names=DISPLAY_NAMES,
    model_colors=MODEL_COLORS,
    out_path=ROOT / "figs/prediction/kdv_pred_error.pdf",
    max_steps=MAX_STEPS,
    ylim=(1e-3, 1.0),
)